In [ ]:
!pip install openai ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 91.4 MB/s eta 0:00:00


In [ ]:
from openai import OpenAI
from ddgs import DDGS
import time

In [ ]:
# 🔐 Replace with your Groq API key
GROQ_API_KEY = "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"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
MODELS = [
    "llama-3.1-8b-instant",
    "llama-3.1-70b-versatile",
    "mixtral-8x7b-32768"
]

def llm_call(prompt, system="You are a helpful AI assistant."):
    for model in MODELS:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7
            )
            return response.choices[0].message.content

        except Exception as e:
            print(f"⚠️ Model {model} failed, trying next...")

    return "❌ All models failed."

In [ ]:
def planner_agent(query):
    prompt = f"""
    Break this query into 3-5 clear research steps.

    Query: {query}

    Only return steps.
    """

    return llm_call(prompt, system="You are a planning agent.")

In [ ]:
def searcher_agent(query, max_results=5):
    results = []

    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append(r['body'])
    except Exception as e:
        print("Search Error:", e)

    return "\n".join(results)

In [ ]:
def analyst_agent(steps, search_data):
    prompt = f"""
    You are a research analyst.

    Steps:
    {steps}

    Data:
    {search_data}

    Analyze deeply and extract meaningful insights.
    """

    return llm_call(prompt, system="You are an expert analyst.")

In [ ]:
def writer_agent(query, analysis):
    prompt = f"""
    Write a structured and clear answer.

    Query:
    {query}

    Analysis:
    {analysis}

    Format:
    - Introduction
    - Key Points
    - Conclusion
    """

    return llm_call(prompt, system="You are a professional writer.")

In [ ]:
def multi_agent_system(query):

    print("🧠 Planner Agent Working...")
    steps = planner_agent(query)
    print("Steps:\n", steps)

    time.sleep(1)

    print("\n🌐 Searcher Agent Working...")
    search_data = searcher_agent(query)

    time.sleep(1)

    print("\n🔍 Analyst Agent Working...")
    analysis = analyst_agent(steps, search_data)

    time.sleep(1)

    print("\n✍️ Writer Agent Working...")
    final_output = writer_agent(query, analysis)

    return final_output

In [ ]:
query = "Latest trends in Generative AI in 2025"

result = multi_agent_system(query)

print("\n✅ FINAL OUTPUT:\n")
print(result)

🧠 Planner Agent Working...
Steps:
 1. Identify Key Sources and Experts
2. Analyze Relevant Research Papers and Studies
3. Explore Industry Conferences and Reports
4. Examine Social Media and Online Forums
5. Consult Academic Databases and Journals

🌐 Searcher Agent Working...

🔍 Analyst Agent Working...

✍️ Writer Agent Working...

✅ FINAL OUTPUT:

**Introduction**

The field of Generative AI has witnessed significant advancements in the past year, with the emergence of new players, increased adoption across various industries, and a notable impact on the job market. As AI continues to transform the way we work, it is essential to stay informed about the latest trends and developments. In this report, we will explore the key insights and recommendations from the latest trends in Generative AI in 2025.

**Key Points**

1. **AI Adoption Patterns**: The past year has seen a significant increase in AI adoption, with younger users driving usage trends. This suggests that AI is becoming incr

In [ ]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 122.8 MB/s eta 0:00:00


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3CotEiKMnoz4q0Yvdsp1eXV0eBh_3CKKekNbuLAp5pMtsiGM4")

In [ ]:
%%writefile app.py
import streamlit as st
from openai import OpenAI
from ddgs import DDGS
import requests
import xml.etree.ElementTree as ET
import time

# -----------------------------
# PAGE CONFIG
# -----------------------------
st.set_page_config(page_title="AI Research Assistant", layout="wide")

# -----------------------------
# UI STYLE
# -----------------------------
st.markdown("""
<style>
.big-title {font-size: 40px; font-weight: bold; color: #4CAF50;}
.section {
    padding: 15px;
    border-radius: 10px;
    margin-bottom: 15px;
    background-color: #1e1e1e;
}
</style>
""", unsafe_allow_html=True)

# -----------------------------
# SIDEBAR
# -----------------------------
st.sidebar.title("⚙️ Settings")

api_key = st.sidebar.text_input("Groq API Key", type="password")

model_choice = st.sidebar.selectbox(
    "Model",
    ["llama-3.1-8b-instant", "llama-3.1-70b-versatile", "mixtral-8x7b-32768"]
)

# -----------------------------
# CLIENT
# -----------------------------
if api_key:
    client = OpenAI(
        api_key=api_key,
        base_url="https://api.groq.com/openai/v1"
    )

# -----------------------------
# LLM FUNCTION
# -----------------------------
def llm_call(prompt, system="You are helpful"):
    try:
        response = client.chat.completions.create(
            model=model_choice,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# -----------------------------
# ARXIV (NO LIBRARY)
# -----------------------------
def fetch_arxiv_papers(query, max_results=5):
    url = f"http://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results={max_results}"

    try:
        response = requests.get(url)
        root = ET.fromstring(response.content)

        papers = []

        for entry in root.findall("{http://www.w3.org/2005/Atom}entry"):
            title = entry.find("{http://www.w3.org/2005/Atom}title").text.strip()
            link = entry.find("{http://www.w3.org/2005/Atom}id").text.strip()

            papers.append((title, link))

        return papers

    except Exception as e:
        return [("Error fetching papers", str(e))]

# -----------------------------
# AGENTS
# -----------------------------

# 🧠 Planner
def planner_agent(query):
    return llm_call(
        f"Break this query into 3-5 research steps:\n{query}",
        "You are a planning agent."
    )

# 🌐 Searcher
def searcher_agent(query):
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=5):
                results.append(r['body'])
    except Exception as e:
        return f"Search Error: {str(e)}"

    return "\n".join(results)

# 🔍 Analyst
def analyst_agent(steps, data):
    return llm_call(
        f"""
        Analyze the following:

        Steps:
        {steps}

        Data:
        {data}

        Extract key insights and trends.
        """,
        "You are an expert analyst."
    )

# ✍️ Writer
def writer_agent(query, analysis):
    return llm_call(
        f"""
        Write a structured answer.

        Query:
        {query}

        Analysis:
        {analysis}

        Format:
        - Introduction
        - Key Points
        - Conclusion
        """,
        "You are a professional writer."
    )

# -----------------------------
# MAIN UI
# -----------------------------
st.markdown('<div class="big-title">🧠 AI Research Assistant</div>', unsafe_allow_html=True)

query = st.text_input("🔍 Enter your research topic")

if st.button("🚀 Run Full Research"):
    if not api_key:
        st.error("Please enter your API key")
    elif not query:
        st.warning("Please enter a query")
    else:
        progress = st.progress(0)

        # 1️⃣ Planner
        st.subheader("🧠 Planner Agent")
        steps = planner_agent(query)
        st.markdown(f'<div class="section">{steps}</div>', unsafe_allow_html=True)
        progress.progress(20)
        time.sleep(1)

        # 2️⃣ Web Search
        st.subheader("🌐 Web Search")
        web_data = searcher_agent(query)
        st.markdown(f'<div class="section">{web_data[:1000]}...</div>', unsafe_allow_html=True)
        progress.progress(40)
        time.sleep(1)

        # 3️⃣ Research Papers
        st.subheader("📚 Related Research Papers")
        papers = fetch_arxiv_papers(query)

        paper_text = ""
        for i, (title, link) in enumerate(papers, 1):
            paper_text += f'{i}. <a href="{link}" target="_blank">{title}</a><br><br>'

        st.markdown(f'<div class="section">{paper_text}</div>', unsafe_allow_html=True)
        progress.progress(60)
        time.sleep(1)

        # 4️⃣ Analysis
        st.subheader("🔍 Analyst Agent")
        analysis = analyst_agent(steps, web_data)
        st.markdown(f'<div class="section">{analysis}</div>', unsafe_allow_html=True)
        progress.progress(80)
        time.sleep(1)

        # 5️⃣ Final Answer
        st.subheader("✍️ Final Answer")
        final = writer_agent(query, analysis)
        st.markdown(f'<div class="section">{final}</div>', unsafe_allow_html=True)
        progress.progress(100)

        st.success("✅ Research Completed!")

Writing app.py


In [ ]:
!pip install streamlit pyngrok openai ddgs requests

In [ ]:
!streamlit run app.py &>/content/logs.txt &

from pyngrok import ngrok
ngrok.connect(8501)

<NgrokTunnel: "https://goldmine-drift-dreaded.ngrok-free.dev" -> "http://localhost:8501">